In [ ]:
import re
import os
import random
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select

import pandas as pd
import numpy as np


In [ ]:
def get_driver(browser=None):

    if browser == "chrome":
        return webdriver.Chrome()

    elif browser == "firefox":
        return webdriver.Firefox()

    elif browser == "edge":
        return webdriver.Edge()

    elif browser == "brave":
        options = Options()
        options.binary_location = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
        return webdriver.Chrome(options=options)

    else:
        raise ValueError("Sem driver no selenium")


driver = get_driver("brave")  
driver.get("http://localhost/prd_sac/sistema/lista_atendimentos.php")
wait = WebDriverWait(driver, 20)

In [ ]:
def iniciar_atendimento():

    print("Atendimento iniciado")

    for i in range(2):      
        print("Loop inicializado")

        select = wait.until(EC.visibility_of_element_located((By.NAME, "resposta_atendente")))   
        print("Select encontrado")

        select_chat = Select(select)
        print("Select chat selecionado")

        if select_chat.options[1:]:
            opcao = random.choice(select_chat.options[1:])
     
            print(f"{opcao} - Selecionada")
            opcao.click()
            print("Opção clicada")

        botao = wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/div[1]/form/button')))
        botao.click()
        

In [ ]:
def extrair_protocolos(driver):
    df_protocolos = []

    tr = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//table/tbody/tr')))
    trs = len(tr)

    for i in range(1, trs + 1):
        col = driver.find_elements(By.XPATH, f'//table/tbody/tr[{i}]/td')

        if len(col) >= 8:
            status_text = col[6].text.strip()

            df_protocolos.append({
                "protocolo": col[0].text,
                "servico": col[2].text,
                "validade": col[4].text,
                "status": col[5].text,
                "feedback": col[6].text
            })

            if status_text == "-":
                
                link = wait.until(EC.element_to_be_clickable((By.XPATH, f'/html/body/div/table/tbody/tr[{i}]/td[8]/a')))   
                url_atend = link.get_attribute("href")
                driver.get(url_atend)
                iniciar_atendimento()
         
                driver.get("http://localhost/prd_sac/sistema/lista_atendimentos.php")
                wait.until(EC.presence_of_all_elements_located((By.XPATH, '//table/tbody/tr')))


    df = pd.DataFrame(df_protocolos)
    print(df)
    return df

extrair_protocolos(driver)